# 示例 01 · LLM 工具调用（ReAct 智能体）

**来源：** [LangGraph 教程 — 构建 ReAct 智能体](https://langchain-ai.github.io/langgraph/tutorials/introduction/)

---

## 学习目标

完成本 Notebook 后，你将能够：

1. 解释 **ReAct（推理 + 行动）** 循环的工作原理
2. 使用 `@tool` 装饰器结合类型注解定义工具
3. 构建包含 `llm_call` 和 `tool_node` 节点的 `StateGraph`
4. 用条件路由控制何时调用工具、何时结束
5. 端到端运行智能体并阅读完整消息轨迹

---

## 背景介绍

### 什么是 ReAct 智能体？

**ReAct 智能体**将*推理*（向 LLM 询问下一步）和*行动*（调用工具并观察结果）
交替进行。循环持续直到 LLM 认为无需再调用工具、可以直接回答为止。

```
开始 → llm_call ──（有工具调用？）──► tool_node ──► llm_call
               └──（无工具调用）──► 结束
```

### LangGraph StateGraph（状态图）

智能体是包含两个节点的**有向图**：

| 节点 | 职责 |
|------|------|
| `llm_call` | 将完整消息历史发送给 LLM，追加其响应 |
| `tool_node` | 执行所有待处理的工具调用，追加结果 |

**状态累积机制：** `AgentState.messages` 使用 `operator.add` 作为归约器——
每个节点的输出会*追加*而非替换，LLM 始终能看到完整的对话历史。

请从上到下依次运行每个单元格。

## 环境配置

In [1]:
import sys
from pathlib import Path

# 将项目根目录加入 sys.path，使 common 包可被导入
_root = Path().resolve().parent.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import operator
from typing import Literal
from typing_extensions import TypedDict, Annotated

from langchain_core.messages import AnyMessage, SystemMessage, ToolMessage, HumanMessage
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END

from common.env import get_env  # noqa: F401 — 触发 .env 加载
from common.llm import create_llm
from common.tracing import create_langfuse_handler, build_langfuse_config, get_langfuse_host

print("✓ 所有依赖导入完成")

✓ 所有依赖导入完成


## 第一步 · 定义工具

工具是用 `@tool` 装饰器修饰的普通 Python 函数。
LangChain 会读取函数的类型注解和文档字符串，自动生成 LLM 决策调用时所需的 JSON Schema。

In [2]:
@tool
def multiply(a: int, b: int) -> int:
    """将 a 乘以 b。"""
    return a * b

@tool
def add(a: int, b: int) -> int:
    """将 a 与 b 相加。"""
    return a + b

@tool
def divide(a: int, b: float) -> float:
    """将 a 除以 b。"""
    return a / b

# 工具名称到工具对象的映射，供 tool_node 按名称分发调用
_tools = [add, multiply, divide]
_tools_by_name = {t.name: t for t in _tools}

print(f"✓ 已定义 {len(_tools)} 个工具：{[t.name for t in _tools]}")

✓ 已定义 3 个工具：['add', 'multiply', 'divide']


## 第二步 · 构建智能体图

以下单元格将四个部分组装在一起：
1. **`AgentState`** — 保存完整消息历史的 TypedDict
2. **节点函数** — `llm_call` 和 `tool_node`
3. **路由函数** — `should_continue` 决定下一个节点
4. **`StateGraph`** — 将所有部分连接并编译为可运行对象

In [3]:
# ── 状态定义 ───────────────────────────────────────────────────────────────
class AgentState(TypedDict):
    # operator.add 表示每个节点的输出追加到列表，而非替换
    messages: Annotated[list[AnyMessage], operator.add]

# ── 绑定工具的模型（只初始化一次，避免重复创建）─────────────────────────
_model = create_llm().bind_tools(_tools)

_SYSTEM_PROMPT = (
    "You are a math assistant that uses tools for arithmetic. "
    "Call one tool at a time and wait for the result before deciding the next step."
)

# ── 节点函数 ───────────────────────────────────────────────────────────────
def llm_call(state: AgentState) -> dict:
    """将完整消息历史传给 LLM 并获取响应。"""
    response = _model.invoke([SystemMessage(content=_SYSTEM_PROMPT)] + state["messages"])
    return {"messages": [response]}

def tool_node(state: AgentState) -> dict:
    """执行最新 AI 消息中的所有工具调用。"""
    results = []
    for tc in state["messages"][-1].tool_calls:
        observation = _tools_by_name[tc["name"]].invoke(tc["args"])
        # 附带工具名称，便于 LLM 在并行调用时匹配结果
        results.append(ToolMessage(content=str(observation), tool_call_id=tc["id"], name=tc["name"]))
    return {"messages": results}

# ── 路由函数 ───────────────────────────────────────────────────────────────
def should_continue(state: AgentState) -> Literal["tool_node", "__end__"]:
    """若最新消息包含待执行的工具调用则继续，否则结束。"""
    return "tool_node" if state["messages"][-1].tool_calls else END

# ── 图构建 ─────────────────────────────────────────────────────────────────
def build_agent():
    g = StateGraph(AgentState)
    g.add_node("llm_call", llm_call)
    g.add_node("tool_node", tool_node)
    g.add_edge(START, "llm_call")
    g.add_conditional_edges("llm_call", should_continue, ["tool_node", END])
    g.add_edge("tool_node", "llm_call")
    return g.compile()

print("✓ 智能体图定义完成，准备编译")

✓ 智能体图定义完成，准备编译


## 第三步 · 运行智能体

智能体将逐步推理：
1. LLM → 调用 `multiply(12345, 17)` → 结果 `209865`
2. LLM → 调用 `divide(209865, 3)` → 结果 `69955.0`
3. LLM → 无更多工具调用 → 返回最终答案

每个 `.pretty_print()` 展示对话链中的一条消息。

In [4]:
agent = build_agent()
question = "What is 12345 multiplied by 17, then divided by 3?"

print(f"问题：{question}")
print("=" * 60)

handler = create_langfuse_handler()
config = build_langfuse_config(
    handler,
    session_id="s_01_notebook_cn",
    user_id="demo-user",
    trace_name="笔记本：数学计算",
)
config["configurable"] = {"thread_id": "nb-s01-cn"}

result = agent.invoke({"messages": [HumanMessage(content=question)]}, config=config)

for msg in result["messages"]:
    msg.pretty_print()

print(f"\n追踪记录：{get_langfuse_host()}")

问题：What is 12345 multiplied by 17, then divided by 3?
================================ Human Message =================================

What is 12345 multiplied by 17, then divided by 3?
================================== Ai Message ==================================

First, I will multiply 12345 by 17.
Tool Calls:
  multiply (call_99fd9b4655374888b3fd55)
 Call ID: call_99fd9b4655374888b3fd55
  Args:
    a: 12345
    b: 17
================================= Tool Message =================================
Name: multiply

209865
================================== Ai Message ==================================

Next, I will divide the result, 209865, by 3.
Tool Calls:
  divide (call_fa07bf9d2d9a479a9cfdd4)
 Call ID: call_fa07bf9d2d9a479a9cfdd4
  Args:
    a: 209865
    b: 3
================================= Tool Message =================================
Name: divide

69955.0
================================== Ai Message ==================================

The result of multiplying 12345 by 1

---

## 总结

| 概念 | API | 用途 |
|------|-----|------|
| 工具定义 | `@tool` 装饰器 | 从类型注解和文档字符串生成 JSON Schema |
| 智能体状态 | `AgentState` + `operator.add` | 累积完整消息历史 |
| LLM 节点 | `llm_call` | 调用模型，返回响应 |
| 工具节点 | `tool_node` | 执行待处理的工具调用，返回结果 |
| 路由 | `should_continue` + `add_conditional_edges` | 决定继续循环还是结束 |
| 图编译 | `StateGraph.compile()` | 生成可运行的智能体 |

### 核心要点

1. **工具就是 Python 函数** — `@tool` 自动完成 Schema 生成
2. **状态只增不减** — `operator.add` 确保任何消息都不会丢失
3. **路由函数是大脑** — 一个条件边控制整个循环
4. **图是可组合的** — 更换 LLM、添加节点或修改路由，不影响其他部分
